# SolarSense: Enhanced Model Benchmark

**Climate Track · Ghotki, Sindh, Pakistan**

This notebook strengthens the first SolarSense experiment by comparing linear and tree-based regression models, using time-series cross-validation on the training period and a final chronological holdout. It uses public NASA POWER observations.

**Target:** next-calendar-day daily solar radiation (kWh/m²/day), not electricity generation.

**Evaluation safeguard:** models are selected using training-only time-series cross-validation. The most recent 20% of supervised observations are held out for final evaluation. Missing days are retained as gaps; values are not interpolated across time.

Run all cells. The notebook creates three CSV outputs for the repository. Report the values it actually produces; do not assume the new models beat the baseline.

In [ ]:
# SolarSense: leakage-aware model benchmark
# Location: Ghotki, Sindh, Pakistan
# Data: NASA POWER daily observations (public data)
# Target: next-calendar-day all-sky surface shortwave radiation (kWh/m²/day)

import warnings
warnings.filterwarnings("ignore")

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Download the same public NASA POWER location and period used in the first run.
LAT, LON = 28.0167, 69.3167
START, END = "20190101", "20251231"
NASA_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMETERS = ["ALLSKY_SFC_SW_DWN", "T2M", "RH2M", "CLOUD_AMT"]

params = {
    "parameters": ",".join(PARAMETERS),
    "community": "RE",
    "longitude": LON,
    "latitude": LAT,
    "start": START,
    "end": END,
    "format": "JSON",
}
response = requests.get(NASA_URL, params=params, timeout=120)
response.raise_for_status()
payload = response.json()
raw = payload["properties"]["parameter"]

data = pd.DataFrame(raw)
data.index = pd.to_datetime(data.index, format="%Y%m%d", errors="coerce")
data.index.name = "date"
data = data.rename(columns={
    "ALLSKY_SFC_SW_DWN": "solar",
    "T2M": "temperature",
    "RH2M": "humidity",
    "CLOUD_AMT": "cloud",
})
required = ["solar", "temperature", "humidity", "cloud"]
missing_columns = [c for c in required if c not in data.columns]
if missing_columns:
    raise ValueError(f"NASA POWER response is missing columns: {missing_columns}")

data = data[required].apply(pd.to_numeric, errors="coerce").replace(-999, np.nan)
data = data.loc[~data.index.isna()].sort_index()
data = data[~data.index.duplicated(keep="last")]

# Reindex to calendar days before shifting, so a data gap cannot become a false
# next-day example. Missing values are NOT interpolated across time.
data = data.asfreq("D")

print(f"NASA POWER rows (calendar days): {len(data):,}")
print(f"Date range: {data.index.min().date()} to {data.index.max().date()}")
print(f"Complete raw rows: {data.dropna().shape[0]:,}")

# 2. Engineer only information available on day t.
# The target is solar radiation on the immediately following calendar day.
target_dates = data.index + pd.Timedelta(days=1)
day_number = target_dates.dayofyear

features = pd.DataFrame(index=data.index)
features["solar_lag_1"] = data["solar"]             # observed solar on day t
features["solar_lag_2"] = data["solar"].shift(1)    # day t-1
features["solar_lag_3"] = data["solar"].shift(2)    # day t-2
features["solar_roll_3"] = data["solar"].rolling(3, min_periods=3).mean()
features["solar_roll_7"] = data["solar"].rolling(7, min_periods=7).mean()
features["temperature"] = data["temperature"]
features["humidity"] = data["humidity"]
features["cloud"] = data["cloud"]
features["temperature_change_1d"] = data["temperature"].diff()
features["cloud_change_1d"] = data["cloud"].diff()
features["target_day_sin"] = np.sin(2 * np.pi * day_number / 365.25)
features["target_day_cos"] = np.cos(2 * np.pi * day_number / 365.25)

FEATURES = list(features.columns)
target = data["solar"].shift(-1).rename("target")
dataset = features.join(target).dropna()

if len(dataset) < 500:
    raise RuntimeError("Too few complete supervised observations to run a reliable benchmark.")

X = dataset[FEATURES]
y = dataset["target"]
baseline = dataset["solar_lag_1"]

# 3. Hold out the latest 20% of dates. No random shuffling.
split_at = int(len(dataset) * 0.80)
X_train, X_test = X.iloc[:split_at], X.iloc[split_at:]
y_train, y_test = y.iloc[:split_at], y.iloc[split_at:]
baseline_test = baseline.iloc[split_at:]

print(f"Supervised observations: {len(dataset):,}")
print(f"Training observations: {len(X_train):,}")
print(f"Held-out test observations: {len(X_test):,}")
print(f"Training period: {X_train.index.min().date()} to {X_train.index.max().date()}")
print(f"Test input dates: {X_test.index.min().date()} to {X_test.index.max().date()}")
print(f"Test target dates: {(X_test.index.min() + pd.Timedelta(days=1)).date()} to {(X_test.index.max() + pd.Timedelta(days=1)).date()}")

# 4. Compare simple and nonlinear models.
# Model choice is made using time-series cross-validation on training data only.
models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression()),
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=250, min_samples_leaf=3, max_features=0.9,
        random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=250, min_samples_leaf=3, max_features=0.9,
        random_state=42, n_jobs=-1
    ),
    "Hist. Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=20, l2_regularization=1.0, random_state=42
    ),
}

time_cv = TimeSeriesSplit(n_splits=4)
rows = []
fitted_models = {}
predictions_by_model = {}

for name, estimator in models.items():
    cv_scores = -cross_val_score(
        clone(estimator), X_train, y_train,
        cv=time_cv, scoring="neg_mean_absolute_error", n_jobs=None
    )
    fitted = clone(estimator)
    fitted.fit(X_train, y_train)
    prediction = np.maximum(0, fitted.predict(X_test))

    rows.append({
        "model": name,
        "training_time_series_cv_mae": float(cv_scores.mean()),
        "test_mae": float(mean_absolute_error(y_test, prediction)),
        "test_rmse": float(np.sqrt(mean_squared_error(y_test, prediction))),
        "test_r2": float(r2_score(y_test, prediction)),
    })
    fitted_models[name] = fitted
    predictions_by_model[name] = prediction

comparison = pd.DataFrame(rows).sort_values("training_time_series_cv_mae")
best_name = comparison.iloc[0]["model"]
best_prediction = predictions_by_model[best_name]

# Persistence is a transparent baseline, not a machine-learning model.
baseline_mae = mean_absolute_error(y_test, baseline_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_test))
best_mae = mean_absolute_error(y_test, best_prediction)
best_rmse = np.sqrt(mean_squared_error(y_test, best_prediction))
best_r2 = r2_score(y_test, best_prediction)
improvement = (baseline_mae - best_mae) / baseline_mae * 100 if baseline_mae > 0 else np.nan

comparison["mae_improvement_vs_persistence_percent"] = (
    (baseline_mae - comparison["test_mae"]) / baseline_mae * 100
)
comparison = comparison.sort_values("training_time_series_cv_mae").reset_index(drop=True)

print("\nMODEL COMPARISON")
display(comparison.round(4))

print("\nFINAL MODEL (selected using training-only time-series CV)")
print(f"Selected model: {best_name}")
print(f"Test MAE:  {best_mae:.4f} kWh/m²/day")
print(f"Test RMSE: {best_rmse:.4f} kWh/m²/day")
print(f"Test R²:   {best_r2:.4f}")
print(f"Persistence MAE: {baseline_mae:.4f} kWh/m²/day")
print(f"MAE improvement vs persistence: {improvement:.2f}%")

if best_mae < baseline_mae:
    print("Result: selected model beat persistence on this held-out period.")
else:
    print("Result: selected model did NOT beat persistence on this held-out period.")

# 5. Visualize the final held-out period and the largest errors.
plot_frame = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Selected model": best_prediction,
    "Persistence": baseline_test.to_numpy(),
}, index=dataset.index[split_at:] + pd.Timedelta(days=1))

plt.figure(figsize=(14, 5))
plt.plot(plot_frame.index, plot_frame["Actual"], label="Actual", linewidth=1.0)
plt.plot(plot_frame.index, plot_frame["Selected model"], label=best_name, linewidth=1.0)
plt.plot(plot_frame.index, plot_frame["Persistence"], label="Persistence baseline", linewidth=0.9, alpha=0.8)
plt.title("SolarSense: Held-out next-day solar radiation forecast")
plt.xlabel("Target date")
plt.ylabel("Solar radiation (kWh/m²/day)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

errors = pd.DataFrame({
    "input_date": dataset.index[split_at:],
    "target_date": dataset.index[split_at:] + pd.Timedelta(days=1),
    "actual": y_test.to_numpy(),
    "predicted": best_prediction,
    "persistence": baseline_test.to_numpy(),
})
errors["absolute_error"] = (errors["actual"] - errors["predicted"]).abs()
print("\nTEN LARGEST MODEL ERRORS")
display(errors.nlargest(10, "absolute_error").round(4))

plt.figure(figsize=(9, 4))
plt.hist(y_test.to_numpy() - best_prediction, bins=30)
plt.title("SolarSense held-out residual distribution")
plt.xlabel("Actual minus predicted (kWh/m²/day)")
plt.ylabel("Number of days")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# 6. Explain the selected model with permutation importance.
# This is descriptive only; it is not used to tune or select the model.
try:
    importance = permutation_importance(
        fitted_models[best_name], X_test, y_test,
        n_repeats=5, random_state=42, scoring="neg_mean_absolute_error"
    )
    importance_table = pd.DataFrame({
        "feature": FEATURES,
        "importance_mae_increase": importance.importances_mean,
    }).sort_values("importance_mae_increase", ascending=False)
    print("\nPERMUTATION IMPORTANCE (higher means more influential in this test set)")
    display(importance_table.round(5))
    plt.figure(figsize=(9, 5))
    top = importance_table.head(10).sort_values("importance_mae_increase")
    plt.barh(top["feature"], top["importance_mae_increase"])
    plt.title("Which inputs mattered most? (permutation importance)")
    plt.xlabel("Increase in MAE when feature is shuffled")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"Permutation-importance calculation was skipped: {exc}")

# 7. Save the reproducible outputs for the repository.
comparison.to_csv("solarsense_model_comparison.csv", index=False)
errors.to_csv("solarsense_enhanced_predictions.csv", index=False)

summary = pd.DataFrame([{
    "selected_model": best_name,
    "training_observations": len(X_train),
    "test_observations": len(X_test),
    "test_mae": best_mae,
    "test_rmse": best_rmse,
    "test_r2": best_r2,
    "persistence_mae": baseline_mae,
    "persistence_rmse": baseline_rmse,
    "mae_improvement_vs_persistence_percent": improvement,
    "test_start": str(X_test.index.min().date()),
    "test_end_input_date": str(X_test.index.max().date()),
}])
summary.to_csv("solarsense_enhanced_summary.csv", index=False)

print("\nSaved:")
print("- solarsense_model_comparison.csv")
print("- solarsense_enhanced_predictions.csv")
print("- solarsense_enhanced_summary.csv")
print("\nIMPORTANT: Report only these newly measured outputs after this notebook finishes.")
print("SolarSense remains an educational prototype, not an operational energy forecast.")
